# Notebook 4: Cross-Lingual Transfer Learning (Hindi → Marathi)
**MarathiMWP Thesis — Chapter 4.5 (Cross-Lingual Transfer) + Chapter 5.3**

Three transfer learning experiments:
1. **Zero-Shot**: Train on HAWP Hindi only → test directly on Marathi
2. **Few-Shot**: Pre-train on HAWP → fine-tune on 10% / 25% / 50% Marathi
3. **Multilingual**: Joint training on Hindi + Marathi data

**Key finding to validate (RQ2)**: Does Hindi transfer improve Marathi accuracy?

**Prerequisites**: Notebooks 01–03 completed, splits/ folder populated.

In [ ]:
import sys, json, random, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from datasets import Dataset as HFDataset
from torch.utils.data import DataLoader as TorchDataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    get_linear_schedule_with_warmup,
)

warnings.filterwarnings('ignore', message='.*use_return_dict.*')

sys.path.insert(0, str(Path('.').resolve()))
from utils.evaluation import compute_metrics, print_metrics, answers_match

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Path('models').mkdir(exist_ok=True)
Path('figures').mkdir(exist_ok=True)

In [ ]:
def load_json(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

marathi_train = load_json('splits/marathi_train.json')
marathi_val   = load_json('splits/marathi_val.json')
marathi_test  = load_json('splits/marathi_test.json')
hawp_train    = load_json('splits/hawp_train.json')
hawp_val      = load_json('splits/hawp_val.json')

gold_test = [x['Equation'] for x in marathi_test]
print(f'Marathi train/val/test: {len(marathi_train)}/{len(marathi_val)}/{len(marathi_test)}')
print(f'HAWP train/val: {len(hawp_train)}/{len(hawp_val)}')

## Helper Functions

In [ ]:
BASE_MODEL  = 'google/mt5-small'
PREFIX      = 'generate equation: '
MAX_SRC_LEN = 128
MAX_TGT_LEN = 64

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
_pad_id   = tokenizer.pad_token_id


def make_dataset(data, prefix=PREFIX):
    src = tokenizer([prefix + x['Problem'] for x in data],
                    max_length=MAX_SRC_LEN, truncation=True, padding=False)
    tgt = tokenizer([x['Equation'] for x in data],
                    max_length=MAX_TGT_LEN, truncation=True, padding=False)
    return HFDataset.from_dict({
        'input_ids'     : src['input_ids'],
        'attention_mask': src['attention_mask'],
        'labels'        : tgt['input_ids'],
    })


def collate(features):
    iids = torch.nn.utils.rnn.pad_sequence(
        [torch.tensor(f['input_ids'], dtype=torch.long) for f in features],
        batch_first=True, padding_value=_pad_id)
    labs = torch.nn.utils.rnn.pad_sequence(
        [torch.tensor(f['labels'], dtype=torch.long) for f in features],
        batch_first=True, padding_value=-100)
    return {'input_ids': iids, 'attention_mask': (iids != _pad_id).long(),
            'labels': labs}


def _run_loop(model, train_ds, val_ds, output_dir,
              n_epochs, lr, batch_size, patience, warmup=100):
    """Shared manual training loop used by train_model and finetune_pretrained."""
    train_dl = TorchDataLoader(train_ds, batch_size=batch_size,
                               shuffle=True,  collate_fn=collate)
    val_dl   = TorchDataLoader(val_ds,   batch_size=32,
                               shuffle=False, collate_fn=collate)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup,
        num_training_steps=len(train_dl) * n_epochs)

    model.to(DEVICE)
    best_val, patience_cnt = float('inf'), 0

    for epoch in range(1, n_epochs + 1):
        model.train()
        tr_loss, n_nan = 0.0, 0
        for batch in train_dl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            loss = model(**batch).loss
            if torch.isnan(loss):
                n_nan += 1
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            tr_loss += loss.item()
        tr_loss /= max(len(train_dl) - n_nan, 1)

        model.eval()
        val_loss, n_nan_v = 0.0, 0
        with torch.no_grad():
            for batch in val_dl:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                l = model(**batch).loss
                if torch.isnan(l):
                    n_nan_v += 1
                else:
                    val_loss += l.item()
        val_loss /= max(len(val_dl) - n_nan_v, 1)

        nan_note = (f' [NaN tr={n_nan} v={n_nan_v}]'
                    if n_nan or n_nan_v else '')
        print(f'  Epoch {epoch:2d}/{n_epochs} | '
              f'Train: {tr_loss:.4f} | Val: {val_loss:.4f}{nan_note}')

        if val_loss < best_val:
            best_val = val_loss
            patience_cnt = 0
            model.save_pretrained(output_dir)
            tokenizer.save_pretrained(output_dir)
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f'  Early stopping at epoch {epoch}')
                break

    return AutoModelForSeq2SeqLM.from_pretrained(
        output_dir, tie_word_embeddings=False)


def train_model(train_data, val_data, output_dir, n_epochs=20, lr=5e-4,
                batch_size=16, patience=4):
    model    = AutoModelForSeq2SeqLM.from_pretrained(
        BASE_MODEL, tie_word_embeddings=False)
    train_ds = make_dataset(train_data)
    val_ds   = make_dataset(val_data)
    return _run_loop(model, train_ds, val_ds, output_dir,
                     n_epochs, lr, batch_size, patience)


def finetune_pretrained(pretrained_dir, finetune_data, val_data,
                        output_dir, n_epochs=10, lr=1e-4):
    model    = AutoModelForSeq2SeqLM.from_pretrained(
        pretrained_dir, tie_word_embeddings=False)
    train_ds = make_dataset(finetune_data)
    val_ds   = make_dataset(val_data)
    return _run_loop(model, train_ds, val_ds, output_dir,
                     n_epochs, lr, batch_size=16, patience=4, warmup=50)


def generate_predictions(model, test_data, prefix=PREFIX, batch_size=32):
    model.eval().to(DEVICE)
    preds = []
    for i in range(0, len(test_data), batch_size):
        batch = test_data[i:i+batch_size]
        texts = [prefix + x['Problem'] for x in batch]
        enc   = tokenizer(texts, return_tensors='pt', padding=True,
                          truncation=True, max_length=MAX_SRC_LEN).to(DEVICE)
        with torch.no_grad():
            ids = model.generate(**enc, max_new_tokens=MAX_TGT_LEN, num_beams=4)
        preds.extend(tokenizer.batch_decode(ids, skip_special_tokens=True))
    return preds

## Experiment 1: Zero-Shot Transfer
Train on HAWP Hindi → test directly on Marathi (no Marathi training data)

In [ ]:
print('=== Experiment 1: Zero-Shot Transfer (HAWP Hindi → Marathi) ===')

zeroshot_model = train_model(
    hawp_train, hawp_val,
    output_dir='models/transfer_zeroshot',
    n_epochs=20,
)

zeroshot_preds   = generate_predictions(zeroshot_model, marathi_test)
zeroshot_metrics = compute_metrics(gold_test, zeroshot_preds)
print_metrics('Zero-Shot Transfer (Hindi → Marathi)', zeroshot_metrics)

## Experiment 2: Few-Shot Transfer
Pre-train on HAWP → fine-tune on 10% / 25% / 50% of Marathi training data

In [ ]:
few_shot_ratios = [0.10, 0.25, 0.50, 1.00]
few_shot_results = {}
few_shot_preds_all = {}

for ratio in few_shot_ratios:
    n_samples = max(10, int(len(marathi_train) * ratio))
    subset    = random.sample(marathi_train, n_samples)

    print(f'\n--- Few-Shot {int(ratio*100)}% ({n_samples} Marathi samples) ---')

    out_dir = f'models/transfer_fewshot_{int(ratio*100)}pct'

    # Fine-tune from the zero-shot Hindi checkpoint
    ft_model = finetune_pretrained(
        pretrained_dir='models/transfer_zeroshot',
        finetune_data=subset,
        val_data=marathi_val,
        output_dir=out_dir,
        n_epochs=15,
        lr=1e-4,
    )

    preds   = generate_predictions(ft_model, marathi_test)
    metrics = compute_metrics(gold_test, preds)
    few_shot_results[f'{int(ratio*100)}%'] = metrics
    few_shot_preds_all[f'{int(ratio*100)}%'] = preds
    print_metrics(f'Few-Shot {int(ratio*100)}% Marathi (Hindi pretrain)', metrics)

## Experiment 3: Multilingual Joint Training
Train on HAWP Hindi + all Marathi simultaneously

In [ ]:
print('=== Experiment 3: Multilingual Joint Training ===')

joint_data = hawp_train + marathi_train
random.shuffle(joint_data)
print(f'Joint training size: {len(joint_data)} problems')

joint_model = train_model(
    joint_data, marathi_val,
    output_dir='models/transfer_multilingual',
    n_epochs=20,
)

joint_preds   = generate_predictions(joint_model, marathi_test)
joint_metrics = compute_metrics(gold_test, joint_preds)
print_metrics('Multilingual Joint (Hindi + Marathi)', joint_metrics)

## Results: Transfer Learning Comparison Table

In [ ]:
# Load monolingual mT5 from Notebook 03 if available
try:
    mono_model = AutoModelForSeq2SeqLM.from_pretrained('models/mt5_marathi')
    mono_preds = generate_predictions(mono_model, marathi_test)
    mono_metrics = compute_metrics(gold_test, mono_preds)
except Exception:
    print('Monolingual mT5 not found — using placeholder')
    mono_metrics = {'answer_accuracy': 0, 'equation_accuracy': 0, 'bleu': 0, 'equation_equivalence': 0, 'n_samples': len(marathi_test)}

transfer_table = {
    'mT5 (Monolingual Marathi)'       : mono_metrics,
    'Zero-Shot Transfer'              : zeroshot_metrics,
    **{f'Few-Shot {k} (Hindi pretrain)': v for k, v in few_shot_results.items()},
    'Multilingual Joint'              : joint_metrics,
}

df_transfer = pd.DataFrame([
    {
        'Model'          : name,
        'Answer Acc (%)'  : f"{m['answer_accuracy']:.1f}",
        'Equation Acc (%)': f"{m['equation_accuracy']:.1f}",
        'BLEU'            : f"{m['bleu']:.1f}",
    }
    for name, m in transfer_table.items()
])

print('\nTransfer Learning Results (Test Set)')
print(df_transfer.to_string(index=False))
df_transfer.to_csv('figures/table_5_3_transfer_results.csv', index=False)

## Transfer Learning Curve (Figure 5.3)

In [ ]:
ratios = [0, 10, 25, 50, 100]
transfer_accs = [
    zeroshot_metrics['answer_accuracy'],
    few_shot_results.get('10%', {}).get('answer_accuracy', 0),
    few_shot_results.get('25%', {}).get('answer_accuracy', 0),
    few_shot_results.get('50%', {}).get('answer_accuracy', 0),
    few_shot_results.get('100%', {}).get('answer_accuracy', 0),
]
mono_accs = [0, 0, 0, 0, mono_metrics['answer_accuracy']]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(ratios, transfer_accs, 'o-', color='#4C72B0', linewidth=2,
        markersize=8, label='Hindi pretrain + Marathi fine-tune')
ax.axhline(y=mono_metrics['answer_accuracy'], color='#DD8452', linestyle='--',
           linewidth=1.5, label=f'Monolingual Marathi (100%)')
ax.axhline(y=zeroshot_metrics['answer_accuracy'], color='gray', linestyle=':',
           linewidth=1.5, label='Zero-Shot (Hindi only)')
ax.set_xlabel('% of Marathi Training Data Used', fontsize=12)
ax.set_ylabel('Answer Accuracy (%)', fontsize=12)
ax.set_title('Few-Shot Transfer Learning Curve\n(Hindi Pretrain → Marathi Fine-Tune)',
             fontsize=13, fontweight='bold')
ax.set_xticks(ratios)
ax.set_xticklabels(['0%\n(zero-shot)', '10%', '25%', '50%', '100%'])
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/fig_5_3_transfer_curve.png', bbox_inches='tight')
plt.show()
print('Saved: figures/fig_5_3_transfer_curve.png')

## Linguistic Analysis: What Transfers Well?

In [ ]:
from utils.data_utils import classify_operation
from utils.evaluation import answers_match

best_preds = few_shot_preds_all.get('100%', joint_preds)

print('Zero-Shot Transfer Accuracy by Operation Type:')
for op in ['addition', 'subtraction', 'multiplication', 'division', 'mixed']:
    indices = [i for i, x in enumerate(marathi_test)
               if classify_operation(x['Equation']) == op]
    if not indices:
        continue
    correct = sum(answers_match(gold_test[i], zeroshot_preds[i]) for i in indices)
    print(f'  {op:<15} : {correct}/{len(indices)} = {correct/len(indices)*100:.1f}%')

print('\nFew-Shot 100% Transfer Accuracy by Operation Type:')
for op in ['addition', 'subtraction', 'multiplication', 'division', 'mixed']:
    indices = [i for i, x in enumerate(marathi_test)
               if classify_operation(x['Equation']) == op]
    if not indices:
        continue
    correct = sum(answers_match(gold_test[i], best_preds[i]) for i in indices)
    print(f'  {op:<15} : {correct}/{len(indices)} = {correct/len(indices)*100:.1f}%')

In [ ]:
# Save all transfer predictions
transfer_preds = [
    {
        'pIndex'        : item['pIndex'],
        'Problem'       : item['Problem'],
        'Gold'          : item['Equation'],
        'ZeroShot'      : zeroshot_preds[i],
        'FewShot50pct'  : few_shot_preds_all.get('50%', [''] * len(marathi_test))[i],
        'FewShot100pct' : few_shot_preds_all.get('100%', [''] * len(marathi_test))[i],
        'Multilingual'  : joint_preds[i],
    }
    for i, item in enumerate(marathi_test)
]
with open('splits/test_predictions_transfer.json', 'w', encoding='utf-8') as f:
    json.dump(transfer_preds, f, ensure_ascii=False, indent=2)
print('Saved: splits/test_predictions_transfer.json')